# 1 Importing Packages

How the *MESSAGE_CA* class works: the class will contain all the functions necessary to run the MESSAGEix Canada model. All that you need to run a model is to:

1. Create your scenario folder under `model/scenarios` with its own `config.toml` file

2. Declare the class reading that scenario, for example: `platform, msg, msg_folder = MESSAGE_CA(scenario_name='baseline')`

In [ ]:
import pandas as pd
from openpyxl import load_workbook

# Load the Excel file
file_path = r"C:\Users\brend\Documents\GitHub\message-ix - CA\model\scenarios\netzero\MESSAGE_Updated_LR_Netzero_test.xlsx"
wb = load_workbook(file_path)

# Ensure at least one sheet is visible
visible_sheets = [sheet for sheet in wb.sheetnames if wb[sheet].sheet_state == 'visible']
if not visible_sheets:
    wb['input'].sheet_state = 'visible'

# Select the 'input' sheet
sheet = wb['emission_factor']

# Convert the sheet to a pandas DataFrame
df = pd.DataFrame(sheet.values)

# Set the first row as column headers
df.columns = df.iloc[0]
df = df[1:]

# Convert 'year_vtg' and 'year_act' to numeric type for the entire DataFrame
df['year_vtg'] = pd.to_numeric(df['year_vtg'], errors='coerce')
df['year_act'] = pd.to_numeric(df['year_act'], errors='coerce')

# Filter for the "technology" column and "h2_atr_ccs" technology
df_filtered = df[df['technology'] == 'h2_atr_ccs']

# Delete rows where 'year_act' is greater than 'year_vtg' + 25
df_filtered = df_filtered[df_filtered['year_act'] <= df_filtered['year_vtg'] + 25]

# Update the original DataFrame with the filtered results
df.update(df_filtered)

# Remove rows that were filtered out
df = df[~((df['technology'] == 'h2_atr_ccs') & (df['year_act'] > df['year_vtg'] + 25))]

# Clear the contents of the 'input' sheet
for row in sheet['A1:' + sheet.dimensions.split(':')[1]]:
    for cell in row:
        cell.value = None

# Write the updated DataFrame back to the Excel file
for r_idx, row in enumerate(df.itertuples(index=False), 1):
    for c_idx, value in enumerate(row, 1):
        sheet.cell(row=r_idx, column=c_idx, value=value)

# Save the workbook
wb.save(file_path)

print("Excel file has been updated and saved.")

In [1]:
import os
import sys


In [4]:
from model.Model import model_MSG 
from model.Plotter import plotter_MSG


# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

In [ ]:
model_MSG.clone_scenario

In [3]:
# Sys and OS Operations

os.chdir("../../")

# Scenario Creation and Set-up

Setting up the MESSAGEix scenario using the scenario configuration file and an input file for technology options in technology.yaml

In [ ]:
# Set-up
msg = model_MSG(scenario_name='baseline')
platform, msg_scenario, msg_folder = msg.setup_scenario()

In [ ]:
df=msg_scenario.par('fix_cost')
h2_atr_data = df[df['technology'] == 'h2_atr']
print(h2_atr_data)



In [7]:
df = msg_scenario.par('growth_activity_lo')
# Apply condition for 'gas_extr_1', 'gas_extr_2', ..., 'gas_extr_6'
df.loc[df['technology'].str.contains(r'gas_extr_[1-6]'), 'value'] = -0.1
# Apply condition for 'gas_extr_mpen'
df.loc[df['technology'].str.contains('gas_extr_mpen'), 'value'] = -0.05


msg_scenario.add_par('growth_activity_lo', df)

In [9]:
df = msg_scenario.par('historical_activity')
df.loc[(df['node_loc'] == 'Alberta') & (df['technology'] == 'gas_extr_mpen'), 'value'] -= 201.505
df.loc[(df['node_loc'] == 'BritishColumbia') & (df['technology'] == 'gas_extr_mpen'), 'value'] -= 116.283
df.loc[(df['node_loc'] == 'Saskatchewan') & (df['technology'] == 'gas_extr_mpen'), 'value'] -= 3.30283

msg_scenario.add_par('historical_activity', df)

In [6]:
msg_scenario.check_out()

In [ ]:
data = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/data/CODERS_data/CER_nuclear.csv')

# Define the region renaming map and desired region format
region_rename_map = {
    'NEWFOUNDLAND AND LABRADOR': 'NewfoundlandandLabrador',
    'BRITISH COLUMBIA': 'BritishColumbia',
    'PRINCE EDWARD ISLAND': 'PrinceEdwardIsland',
    'NORTHWEST TERRITORIES': 'NorthwestTerritories',
    'YUKON': 'Yukon',
    'NOVA SCOTIA': 'NovaScotia',
    'NEW BRUNSWICK': 'NewBrunswick',
    'ONTARIO': 'Ontario',
    'QUEBEC': 'Quebec',
    'ALBERTA': 'Alberta',
    'SASKATCHEWAN': 'Saskatchewan',
    'MANITOBA': 'Manitoba',
    'NUNAVUT': 'Nunavut'
}

# Rename regions
data['region'] = data['region'].str.upper().map(region_rename_map)

# Filter for 5-year timesteps starting at 2020
# Replace 2023 with 2020 and exclude existing 2020 entries
data = data[(data['year'] % 5 == 0) | (data['year'] == 2023)]
data.loc[data['year'] == 2023, 'year'] = 2020

# Exclude existing 2020 entries that weren't modified
data = data.drop_duplicates(subset=['region', 'year', 'scenario', 'source'], keep='last')

data.rename(columns={'source': 'technology', 'region': 'node_loc', 'year': 'year_act'}, inplace=True)
data.drop(columns = ['scenario','dataset'], inplace=True)
data['technology'] = 'nuc_lc'
data['value'] = data['value'] / (8.76*3.6)
data['unit'] = 'GWa/a'
data['mode'] ='M1'
data['time'] = 'year'

# Step 1: Filter the dataset for the year 2050 to get the reference values
data_2050 = data[data['year_act'] == 2050]

# Step 2: Create new rows for 2055 and 2060 using the 2050 values
data_extended = pd.concat([
    data,
    data_2050.assign(year_act=2055),
    data_2050.assign(year_act=2060)
])

# Step 3: Sort the extended dataset by `node_loc` and `year_act` for clarity
data_extended = data_extended.sort_values(by=['node_loc', 'year_act']).reset_index(drop=True)

# msg_scenario.check_out()
data_hist= data[data['year_act'] <= 2020]
msg_scenario.add_par('historical_activity', data_hist)
data_future = data[data['year_act'] >= 2020]
msg_scenario.add_par('bound_activity_lo', data_extended)

In [5]:
import pandas as pd
import pyam as pyam
import message_ix as message_ix

In [6]:
msg_scenario.remove_solution()

In [20]:
caseName = msg_scenario.model + '__' + msg_scenario.scenario + '__v' + str(msg_scenario.version)
msg_scenario.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

In [ ]:
# msgpes = msg_scenario.clone(msg_scenario.model, 'netzero_pes',
#                      keep_solution=False)
df = pd.read_excel(
    r'C:\Users\brend\Documents\GitHub\message-ix - CA\Inv_cost_low_LR.xlsx')
msg_scenario.check_out()
msg_scenario.add_par('inv_cost', df)
msg_scenario.commit('Decreasing costs of innovative techs by 10%')
# Exporting the built model (Scenario) to GAMS with an optional case name
caseName = msg_scenario.model + '__' + msg_scenario.scenario + '__v' + str(msgpes.version)
msg_scenario.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

In [ ]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

shares = ['share_h2_fc_i', 'share_h2_i', 'share_h2_fc_RC', 'share_h2_rc']

years = list(range(2025, 2061, 5))

# Get the entire share_commodity_lo parameter once
share_df = msg_scenario.par('share_commodity_lo')

# Rename columns to match expected dimensions
share_df = share_df.rename(columns={'node_share': 'node', 'year_act': 'year'})

# Print information about the DataFrame
print("\nDataFrame Info:")
share_df.info()

print("\nUnique values in 'shares' column:")
print(share_df['shares'].unique())

print("\nUnique values in 'node' column:")
print(share_df['node'].unique())

print("\nUnique values in 'year' column:")
print(share_df['year'].unique())

print("\nUnique values in 'time' column:")
print(share_df['time'].unique())

# Create a mask for the rows we want to modify
mask = (share_df['shares'].isin(shares)) & \
       (share_df['node'].isin(provinces)) & \
       (share_df['year'].isin(years)) & \
       (share_df['time'] == 'year')

# Print the number of True values in each part of the mask
print("\nNumber of True values in each part of the mask:")
print("shares mask:", share_df['shares'].isin(shares).sum())
print("node mask:", share_df['node'].isin(provinces).sum())
print("year mask:", share_df['year'].isin(years).sum())
print("time mask:", (share_df['time'] == 'year').sum())

# Multiply the values for the masked rows by 3
share_df.loc[mask, 'value'] *= 2

# Print summary to verify changes
print("\nSummary of modifications:")
print(share_df[mask]['value'].describe())

# Print the number of modified rows
print(f"\nNumber of rows modified: {mask.sum()}")

# Print unique values in 'shares' column to verify which shares were affected
print("\nShares affected:")
print(share_df[mask]['shares'].unique())

# Remove the existing parameter
msg_scenario.remove_par('share_commodity_lo')

msg_scenario.init_par('share_commodity_lo', ['shares', 'node', 'year', 'time'])

share_df = share_df.rename(columns={'node': 'node_share', 'year': 'year_act'})

# Add the modified parameter back to the scenario
msg_scenario.add_par('share_commodity_lo', share_df)

In [ ]:


'''years = list(range(2025, 2041, 5))
technology = "h2_atr_ccs"
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Loop through provinces and years to remove the 'input' parameter
for province in provinces:
    for year_act in years:
        for year_vtg in years:
            if year_vtg <= year_act <= year_vtg + 10:
                msg_scenario.remove_par('input', 
                    {
                        'node_loc': province,
                        'technology': technology,
                        'year_vtg': year_vtg,
                        'year_act': year_act,
                        'mode': "M1",
                        'node_origin': province,
                        'commodity': ["electr"],
                        'level': ["primary"],
                        'time': "year",
                        'time_origin': "year"
                    }
                )'''

In [ ]:
'''years = list(range(2020, 2041, 5))
technology = "h2_smr_ccs"
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Loop through provinces and years to remove the 'input' parameter
for province in provinces:
    for year_act in years:
        for year_vtg in years:
            if year_vtg <= year_act <= year_vtg + 25:
                msg_scenario.remove_par('emission_factor', 
                    {
                        'node_loc': province,
                        'technology': technology,
                        'year_vtg': year_vtg,
                        'year_act': year_act,
                        'mode': "M1",
                        'emission': ["CO2_Emission"],
                    }
                )'''

In [ ]:
'''import pandas as pd

# Define the provinces and the years
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2030, 2061, 5))  # 2030, ..., 2060
years2 = list(range(2040, 2061, 5)) 
technologies = ['h2_atr_ccs', 'meth_h2']
technology2 = 'h2_gas_fc_trp'
value = 0.015
value2= 0.12162

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('initial_activity_up', 
                        [province, technology, year, "year"], 
                        value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years2:
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology2, year, "year"], 
                    value2, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed       
    for technology in technologies:      
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology, "2020", "year"], 
                    0.006, 'GWa')
    for technology in technologies:
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology, "2025", "year"], 
                    0.0105, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2025", "year"], 
                0.10815, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2030", "year"], 
                0.11295, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2035", "year"], 
                0.1173, 'GWa')'''



In [7]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']
years = list(range(2020, 2061, 5)) 
value = 62.5

for province in provinces:
    for year_vtg in years:
        for year_act in years:
            if year_vtg <= year_act <= year_vtg + 25:
                msg_scenario.add_par('fix_cost', 
                            [province, "h2_fc_trp", year_vtg, year_act], 
                            value, 'GWa') 

In [ ]:
'''value = 0.08
value2= 0.075
years = list(range(2020, 2061, 5)) 

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('growth_activity_up', 
                        [province, technology, year, "year"], 
                        value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
        
        msg_scenario.add_par('growth_activity_up', 
                    [province, technology2, year, "year"], 
                    value2, 'GWa')'''

In [ ]:
'''value = 0.006
#h2_atr
technologies= ['h2_atr_ccs', 'meth_h2', 'h2_atr'] #meed to do for both w and without ccs

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('initial_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
            
'''

In [ ]:
'''value = 0.05 #meed to do for both w and without ccs
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('growth_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed'''

In [ ]:
'''value = 0.5 #meed to do for both w and without ccs
#h2_atr
technologies= ['h2_atr_ccs', 'h2_atr']
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('level_cost_activity_soft_lo', 
                    [province, technology, year, "year"], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
            msg_scenario.add_par('level_cost_activity_soft_up', 
                    [province, technology, year, "year"], 
                    value, 'GWa')'''

In [ ]:
'''value = -0.05 #meed to do for both w and without ccs
#h2_atr
technologies= ['h2_atr_ccs', 'h2_atr']
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('soft_activity_lo', 
                    [province, technology, year, "year"], 
                    value, 'GWa') 
            
value = 0.05 #meed to do for both w and without ccs

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('soft_activity_up', 
                    [province, technology, year, "year"], 
                    value, 'GWa')''' 

In [ ]:
'''msg_scenario.add_set("cat_tec", ["transformation_fossil", "h2_atr_ccs"])

msg_scenario.add_set("cat_tec", ["powerplant", "h2_atr_ccs"])'''

In [ ]:
'''msg_scenario.add_set("cat_tec", ["h2_trp", "h2_gas_fc_trp"])

msg_scenario.add_set("cat_tec", ["transport", "h2_gas_fc_trp"])

# Optionally, you can check if the addition was successful
cat_tec = msg_scenario.set("cat_tec")
print(cat_tec)'''

In [8]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']
years = list(range(2020, 2061, 5)) 
value = -0.62319
value2= 1

for province in provinces:
    for year in years:
      msg_scenario.add_par('relation_activity', 
                    ["h2_scrub_limit", province, year, province, "h2_atr_ccs", year, "M1"], 
                    value, 'GWa/a') 
      
      msg_scenario.add_par('relation_activity', 
                    ["h2_exp_limit", province, year, province, "h2_atr_ccs", year, "M1"], 
                    value2, 'GWa/a')
      msg_scenario.add_par('relation_activity', 
                    ["h2_exp_limit", province, year, province, "h2_atr", year, "M1"], 
                    value2, 'GWa/a') 

In [8]:
msg_scenario.commit("added parameters/sets")

In [7]:
msg_scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"},
				)

In [9]:

"""
Apply the Output-Based Pricing System (OBPS) policy for the Baseline Canada Scenario
to the given MESSAGEix scenario.

This function modifies the scenario's fixed costs based on emissions intensity and carbon pricing.

Args:
    scenario (ixmp.Scenario): The MESSAGEix scenario to apply the OBPS policy to.
    data_file_path (str): Path to the Excel file containing necessary data.
"""

# Read the required data
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
            'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
            'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Define conditions for filtering the data
conditions_2 = (
    data_relation['technology'].isin([
        'oil_extr_1', 'oil_extr_2', 'oil_extr_3', 'oil_extr_4', 'oil_extr_5', 'oil_extr_6',
        'cement_CO2', 'coal_extr', 'gas_extr_1', 'gas_extr_2', 'gas_extr_3', 'gas_extr_4', 
        'gas_extr_5', 'gas_extr_6', 'h2_smr_ccs', 'h2_smr', 'h2_atr', 'h2_atr_ccs'
    ]) &
    (data_relation['emission'] == 'CO2_cc') &
    (data_relation['node_loc'].isin(provinces))
)

# Filter the data based on the conditions
result = data_relation[conditions_2][['technology', 'value', 'node_loc', 'year_act']]

# Initialize columns for calculations
result['Emission Intensity Reduction_baseline'] = 0.0

# Calculate Emission Intensity Reduction_baseline
for index, row in result.iterrows():
    if 2020 < row['year_act'] < 2055:
        previous_year_value = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == row['year_act'] - 5), 
            'Emission Intensity Reduction_baseline'
        ].values
        if len(previous_year_value) > 0:
            result.loc[index, 'Emission Intensity Reduction_baseline'] += previous_year_value[0] + 3.33/100

# Calculate Emission intensity standard_Baseline Canada Scenario
result['Emission intensity standard_Baseline Canada Scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        value_cal = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2015), 
            'value'
        ]
        if not value_cal.empty:
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]
        else:
            value_cal = result.loc[
                (result['technology'] == row['technology']) & 
                (result['year_act'] == 2020), 
                'value'
            ]
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]

# Calculate Payable emission intensity_Baseline scenario
result['Payable emission intensity_Baseline scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        result.loc[index, 'Payable emission intensity_Baseline scenario'] = row['value'] - row['Emission intensity standard_Baseline Canada Scenario']

# Calculate carbon cost_Baseline Canada scenario
rate = 0.73
result['carbon cost_Baseline Canada scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] == 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 125
    elif row['year_act'] > 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 170

# Adjust for years 2055 and 2060
for index, row in result.iterrows():
    if row['year_act'] > 2050:
        value_to_assign = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2050), 
            'carbon cost_Baseline Canada scenario'
        ].values
        if len(value_to_assign) > 0:
            result.at[index, 'carbon cost_Baseline Canada scenario'] = value_to_assign[0]

# Merge with fix_cost data and update the scenario's fix_cost
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')
# Read the fix_cost data
data_fixcost = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='fix_cost')

merged_data = pd.merge(
    data_fixcost, 
    result[['technology', 'year_act', 'node_loc', 'carbon cost_Baseline Canada scenario']],
    left_on=['technology', 'year_act', 'node_loc'],
    right_on=['technology', 'year_act', 'node_loc'],
    how='left'
)
merged_data['value'] = merged_data['value'] + merged_data['carbon cost_Baseline Canada scenario'].fillna(0)
merged_data.drop(columns=['carbon cost_Baseline Canada scenario'], inplace=True)

# Assuming 'merged_data' contains the data you've shown

for index, row in merged_data.iterrows():
    msg_scenario.add_par('fix_cost', pd.DataFrame({
        'node_loc': [row['node_loc']],
        'technology': [row['technology']],
        'year_vtg': [row['year_vtg']],
        'year_act': [row['year_act']],
        'mode': ['M1'],  # Assuming 'standard' mode for all entries
        'unit': [row['unit']],
        'value': [row['value']]
    }, index=[0]))

In [5]:
msg_scenario.to_excel(r"C:\Users\brend\Documents\GitHub\message-ix - CA\model\notebooks\model_default_netzero_v5.xlsx")


In [5]:
import message_ix
import pandas as pd
import pyam

In [ ]:
output_path=R"C:\Users\brend\Documents\GitHub\message-ix - CA\output\new_reporting_dec6\baseline.csv"
min_year= 2015
max_year= 2060
from_excel=False
unit_correct=True

rep = message_ix.Reporter.from_scenario(
    msg.scenario,
    units={"apply": {"var_cost": "", "inv_cost": "", "fix_cost": ""}},
)
report = rep.get("message::default")

if from_excel:
    report_df = pd.read_csv("MESSAGE_NO_example_default_report.csv")
else:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

years = msg.scenario.set("year").tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

report_df.fillna(0, inplace=True)

report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(
    report_df.columns.to_series()["2020" : str(min_year)], axis=1
)
report_df.to_csv('test.csv')
in_flows = report_df[report_df["Variable"].str.startswith("in|")]
out_flows = report_df[report_df["Variable"].str.startswith("out|")]
all_flows = pd.concat([out_flows, in_flows])

report_df["Model"] = "MESSAGEix-Canada"
report_df["Scenario"] = "NZ-PES"
report_df["Variable"] = report_df["Variable"].str.replace(
    "emis", "Emissions", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "total om cost|", "Total Costs|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP_NEW|new capacity|", "Capacity Addition|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP|capacity|", "Capacity|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "inv cost|", "Capital Cost|", regex=False
)
report_df["Region"] = report_df["Region"].apply(lambda x: x.split("|")[0])

report_df.columns = report_df.columns.str.strip()  # Remove any leading/trailing whitespace
report_df.rename(columns={'Model': 'model', 'Scenario': 'scenario', 'Variable': 'variable'}, inplace=True)

report_df = report_df.drop_duplicates(subset=["model", "scenario", "Region", "variable"], keep='first')


pyam_df = pyam.IamDataFrame(report_df)

variable = pyam_df.variable
pyam_df.aggregate_region(variable, append=True)
pyam_df.rename({"region": {"World": "Canada"}}, inplace=True)

pyam_df.to_csv(output_path)

print(f"Report saved to {output_path}")



In [ ]:
import os

output_dir = r'C:\Users\brend\Documents\Github\message-ix - CA\scenarios\baseline'

# Check if the directory exists, if not, create it
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'report.csv')

# Now save the file
pyam_df.to_csv(output_path)
print(f"Report saved to {output_path}")

In [ ]:
print(report_df["Region"].value_counts())

In [ ]:
report_df[report_df["variable"] =='Secondary Energy|electr|h2_smr_ccs|M1']


In [ ]:
print(report_df["variable"].value_counts())

In [ ]:
report_df.columns

In [ ]:
test = report_df['variable'].value_counts()
# print("Number of unique variables:", report_df['Variable'].nunique())


In [6]:
#from message_ix_models.tools.costs.config import Config
#from message_ix_models.tools.costs.projections import create_cost_projections

In [ ]:
'''# Step 1: Define hydrogen-related technologies
hydrogen_tech_list = ["h2_atr", "h2_atr_ccs"]

# Step 2: Define configuration for cost projections using the 'convergence' method
cfg = Config(
    module="energy",  # Assuming hydrogen falls under energy module
    method="convergence",  # Use convergence method for cost projections
    base_year=2020,  # Base year for the cost projections
    convergence_year=2050,  # Year to which costs should converge
    node="R12",  # Set your region, e.g., R12 (or your specific region code)
    ref_region="R12_NAM"  # Set the reference region
)

# Step 3: Generate base cost projections for all energy technologies
cost_projections = create_cost_projections(cfg)

# Step 4: Extract investment and fixed cost DataFrames
inv_cost_df = cost_projections["inv_cost"]
fix_cost_df = cost_projections["fix_cost"]

# Print columns for debugging
print("Investment Cost DataFrame Columns:")
print(inv_cost_df.columns)

print("\nFixed Cost DataFrame Columns:")
print(fix_cost_df.columns)

# Step 5: Filter cost projections for hydrogen-related technologies
hydrogen_inv_cost = inv_cost_df[inv_cost_df['technology'].isin(hydrogen_tech_list)]

# Debugging: Print the filtered DataFrame to understand its structure
print("\nFiltered Hydrogen Investment Cost DataFrame:")
print(hydrogen_inv_cost.head())

# Check if 'year' column is present
if 'year' not in hydrogen_inv_cost.columns:
    print("No 'year' column found in hydrogen investment costs. Available columns are:", hydrogen_inv_cost.columns)

# Step 6: Apply the learning rate manually to the investment cost
learning_rate = 0.1  # 10% learning rate

# Use 'year' instead of 'year_act' for base year costs
base_year_costs = hydrogen_inv_cost[hydrogen_inv_cost['year'] == 2020].set_index('technology')['inv_cost']

# Apply learning rate decay to future years
for year in hydrogen_inv_cost['year'].unique():
    if year > 2020:
        for tech in hydrogen_tech_list:
            # Calculate the new cost for each technology
            if tech in base_year_costs.index:
                new_cost = base_year_costs[tech] * (1 - learning_rate) ** ((year - 2020) / 5)
                # Update the DataFrame
                hydrogen_inv_cost.loc[
                    (hydrogen_inv_cost['year'] == year) & (hydrogen_inv_cost['technology'] == tech), 'inv_cost'] = new_cost

# Step 7: Extract final investment and fixed costs
inv_cost = hydrogen_inv_cost[['technology', 'year', 'inv_cost']] if 'inv_cost' in hydrogen_inv_cost.columns else None
fix_cost = fix_cost_df[fix_cost_df['technology'].isin(hydrogen_tech_list)] if 'technology' in fix_cost_df.columns else None

# You can now apply these adjusted costs to your MESSAGEix model.

# Print adjusted costs for verification
print("\nAdjusted Investment Costs:")
print(inv_cost)'''

In [5]:
msg.scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"})

In [18]:
msg_plt = Plotter_MSG(msg_scenario)
nodes = msg_scenario.set('node')[2:]


In [ ]:
msg_plt.plotter(case="test",nodeloc=nodes,path="output",yr_min=2020, yr_max=2060)